In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from hyperparameters_config import DEVICE, BATCH_SIZE, SEQ_LEN , NHEAD, NUM_LAYERS, MASK_RATIO, D_MODEL, CODEBOOK_SIZE
from modelMASKGIT import MotionMaskGIT

In [2]:
# Fake motion token to create VQVAE --> NEED TO BE REMOVED BEFORE THE WORKING ON REAL DATA

tokens = torch.randint( #### NEED TO BE CHANGED OR REMOVE WHEN WORKING WITH REAL DATA
    0,
    CODEBOOK_SIZE,
    (BATCH_SIZE, SEQ_LEN)
).to(DEVICE)

targets = tokens.clone()

#rand mask

mask = (
    torch.rand(
        BATCH_SIZE,
        SEQ_LEN
    ) < MASK_RATIO
).to(DEVICE)

masked_tokens = tokens.clone()

MASK_TOKEN_ID = CODEBOOK_SIZE

masked_tokens[mask] = MASK_TOKEN_ID

#run model

model = MotionMaskGIT(
    codebook_size=CODEBOOK_SIZE,
    d_model=D_MODEL,
    nhead=NHEAD,
    num_layers=NUM_LAYERS
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)

logits = model(masked_tokens)

print("Input tokens:", tokens.shape)
print("Masked tokens:", masked_tokens.shape)
print("Logits:", logits.shape)

#Loss only on masked position

masked_logits = logits[mask]
masked_targets = targets[mask]

loss = F.cross_entropy(
    masked_logits,
    masked_targets
)

#BP

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("\nLoss:", loss.item())

# MOCK PREDICTION --> AGAIN, CHANGE ABOVE TO USE WITH REAL DATA

predictions = logits.argmax(dim=-1)

print("\nOriginal:")
print(tokens[0][:20])

print("\nMasked:")
print(masked_tokens[0][:20])

print("\nPredicted:")
print(predictions[0][:20])

Input tokens: torch.Size([4, 100])
Masked tokens: torch.Size([4, 100])
Logits: torch.Size([4, 100, 1024])

Loss: 7.1024980545043945

Original:
tensor([703,  18, 547, 497, 592, 174, 319, 815,  44,   5, 631, 594, 823, 542,
         61, 180, 179, 693, 666, 596])

Masked:
tensor([ 703,   18, 1024,  497, 1024, 1024,  319,  815,   44, 1024, 1024,  594,
         823, 1024,   61,  180,  179,  693, 1024,  596])

Predicted:
tensor([469,  91,  91, 508,  91, 111, 281, 270, 342,  91,  91,  28, 550,  91,
         28, 622,  91,  28,  91,  91])
